In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

MODEL_PATH = "pose_landmarker.task"

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    min_pose_detection_confidence=0.1,
    min_pose_presence_confidence=0.1,
    min_tracking_confidence=0.1
)
detector = vision.PoseLandmarker.create_from_options(options)



I0000 00:00:1775578618.003878 2020067 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775578618.042741 2020070 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775578618.087278 2020070 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [2]:
models = [
    "qwen2.5vl:7b",
    'gemma3:12b',
    "gemma4:e4b",
    "llava:7b"
    ]
    

In [3]:
from utils import *

In [4]:
import ollama
import time
import pandas as pd
from pydantic import BaseModel, Field

In [5]:
class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str = Field(
        min_length=20,
        max_length=300,
        description="Brief coaching feedback in 2-4 sentences."
    )

In [6]:
throwprompt2 = """
You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
"""

throwprompt3 = """
You are a discus coach whose job is to identify the single biggest flaw in a throw.

Look across the full sequence of images and determine:
1. The overall score from 1-10
2. The biggest mistake hurting the throw
3. The one change that would improve the throw the most

Keep the feedback concise and concrete. Do not mention uncertainty unless absolutely necessary.
"""

throwprompt4 = """
You are an AI sports judge trained to score discus technique from still frames.

You are seeing multiple frames sampled throughout one throw. Infer the movement between frames.

Score the throw from 1-10 using this rubric:
- 1-3: major balance, timing, or release problems
- 4-6: decent throw with noticeable flaws
- 7-8: technically good throw with minor issues
- 9-10: highly polished, competitive form

In your feedback, mention at least:
- one thing the athlete did well
- one thing to improve
- what phase of the throw looked weakest
"""

throwprompt5 = """
You are a brutally honest but helpful throwing coach.

Analyze the series of images as if you are watching a video of the throw. Pay close attention to:
- whether the athlete stays balanced
- whether the hips lead the shoulders
- whether the throwing arm lags correctly
- whether the release appears high and powerful

Give a score from 1-10. Then provide feedback in the style of a coach talking directly to the athlete: short, blunt, and practical.
"""

In [7]:


def critique_throw(image_paths: list,
                   mod: str = "llava:7b",
                   prompt: str = throwprompt2,  
                   temp: float = 0.0):
    
    resp = ollama.chat(
        model=mod,
        messages=[
            {
                "role": "system",
                "content": prompt,
                "images": image_paths
            }
        ],
        format=ThrowCritique.model_json_schema(),
        options={"temperature": temp}
    )

    raw = resp["message"]["content"]
    return ThrowCritique.model_validate_json(raw)

In [8]:
prompts=[throwprompt2, throwprompt3,throwprompt4, throwprompt5]

In [9]:
def doscore(count, instructions, mod):
    
    scores=[]
    durations=[]
    feedbacks=[]
    names=[]
    fileframeslist=[]
    filefpslist=[]
    secondslist=[]
    
    for i in range(1, 8):
        
        idstr = "" if i == 1 else f"-{i}"
        file = f"videoplayback{idstr}.mp4"
        
        fileframes=get_frame_count(file)
        fileframeslist.append(fileframes)

        filefps=get_fps(file)
        filefpslist.append(filefps)

        secondslist.append(fileframes/filefps)
        
        start = time.time()

        try:
            frames = sample_frames_from_mp4(file, count)
            overs = pose_overlays_from_frames(frames, detector)
            img_paths = write_temp_images_for_llm(overs)
            crit = critique_throw(image_paths=img_paths, mod=mod, prompt=instructions)
            scores.append(crit.score)
            feedbacks.append(crit.feedback)
            names.append(file)
            
        except Exception as e:
            print(f"Failed on {file}: {e}")
            scores.append(pd.NA)
            feedbacks.append(pd.NA)
            names.append(file)

        durations.append(time.time() - start)

    return (
        pd.DataFrame({"score": scores, "duration": durations,
                      "feedback":feedbacks,"filenames":names,
                      "fileframes":fileframeslist,"filefps":filefpslist,
                     "fileseconds":secondslist}).\
            assign(frames=count, prompt=instructions[:20], model=mod)
    )

In [10]:
report=pd.DataFrame()
for i in range(12,19,3):
    for j in prompts:
        for k in models:
            df=doscore(i,j,k)
            report=pd.concat([report,df])
            print(i,k)

W0000 00:00:1775578618.787023 2020069 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


12 qwen2.5vl:7b
12 gemma3:12b
12 gemma4:e4b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 gemma4:e4b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 gemma4:e4b
12 llava:7b
12 qwen2.5vl:7b
12 gemma3:12b
12 gemma4:e4b
12 llava:7b
Failed on videoplayback.mp4: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed
See: https://github.com/ggml-org/llama.cpp/pull/17869 (status code: 500)
Failed on videoplayback-4.mp4: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed
See: https://github.com/ggml-org/llama.cpp/pull/17869 (status code: 500)
Failed on videoplayback-6.mp4: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed
See: https://github.com/ggml-org/llama.cpp/pull/17869 (status code: 500)
Failed on videoplayback-7.mp4: an error was encountered while running the model: GGML_ASSERT(a->ne[2] * 4 == b->ne[0]) failed
See: https://github.com/ggml-org/llama.cpp/pull/17869 (

In [11]:
report.to_csv("wannoreport2.csv")

In [12]:
import os
import time

while True:
    os.system('say "I am done now"')
    time.sleep(2)  # wait 2 seconds before saying it again

KeyboardInterrupt: 